In [ ]:
import pandas as pd
import json
import time
import os
from openai import OpenAI
from tqdm import tqdm

API_KEY = "c52fb94b73644f5fa7279b64ad9e6234.ihtDd5UAAxsmaOwH"                    
BASE_URL = "https://open.bigmodel.cn/api/paas/v4"
MODEL_NAME = "glm-4-flash"                      

INPUT_FILE = "cleaned_fraud_data.xlsx"          
OUTPUT_FILE = "llm_labeled_by_zhipu.xlsx"       

TEXT_COLUMN = "content"                         
TITLE_COLUMN = "title"                          

SAVE_EVERY = 10

SLEEP_TIME = 0.8


TEST_MODE = False
# ================================================================

# 初始化客户端
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def build_prompt(text, title=""):
    """
    构建发送给 LLM 的提示词，要求输出 JSON 格式的三级分类结果。
    """
    prompt = f"""
你是一个诈骗案件分析专家。请根据以下帖子内容，严格按照给定的三级分类标准进行标注。

帖子标题：{title if title else "无"}
帖子正文：{text}

【一级维度：核心损失结果类型】四选一：
- 直接金钱损失型：受害者自有资金直接转给骗子，无借贷、无后续债务。
- 间接财产损失型：诱导受害者借贷、刷信用卡、产生债务或征信问题。
- 情感/精神损失型：以情感操控、PUA为主，无明显财产损失。
- 复合多重损失型：同时包含上述两种及以上损失类型。

【二级维度：危害程度等级】四选一：
- 轻度危害：损失≤3000元，可追回，无持续精神影响，无征信问题。
- 中度危害：损失3000~50万元，无法全额追回，或有短期情绪影响，或借贷几万元。
- 重度危害：损失≥50万元，或精神影响超过3个月且就医，或借贷≥10万，或多种损失复合。
- 极端危害：损失≥500万元，或出现自杀/重伤/精神失常，或家破人亡。

【三级维度：损失时间周期】二选一：
- 即时一次性损失：一次性被骗，后续无持续损害，影响≤1个月。
- 长期持续损失：损失持续发生（如长期还债、长期抑郁、征信长期异常），影响超过3个月。

请仔细阅读内容，即使语言是粤语或英语，也要准确理解。然后只返回一个JSON对象，格式如下：
{{
  "loss_type": "直接金钱损失型",
  "severity": "中度危害",
  "time_span": "即时一次性损失",
  "brief_reason": "一句话说明判断依据"
}}

不要包含任何其他文字或解释，只输出JSON。
"""
    return prompt

def call_llm(prompt, max_retries=3):
    """
    调用智谱 API，支持重试和 JSON 解析。
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "你是一个专业的诈骗类型标注助手，必须只输出一个合法的JSON对象，不要包含任何其他文字。"},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                
                response_format={"type": "json_object"}
            )
            result_text = response.choices[0].message.content.strip()
            
            if result_text.startswith("```json"):
                result_text = result_text[7:]
            elif result_text.startswith("```"):
                result_text = result_text[3:]
            if result_text.endswith("```"):
                result_text = result_text[:-3]
            result_text = result_text.strip()
            return json.loads(result_text)
        except Exception as e:
            print(f"\n调用出错 (尝试 {attempt+1}/{max_retries}): {e}")
            time.sleep(3)
    
    return {"loss_type": "标注失败", "severity": "标注失败", "time_span": "标注失败", "brief_reason": "API调用失败"}

def load_or_create_df():
    """
    加载数据，并自动合并已有标注进度（断点续跑核心逻辑）。
    """
    df_raw = pd.read_excel(INPUT_FILE)
    print(f"原始数据共 {len(df_raw)} 条。")

    # 初始化标注列
    for col in ["loss_type", "severity", "time_span", "llm_reason"]:
        if col not in df_raw.columns:
            df_raw[col] = ""

    # 如果输出文件已存在，读取并合并已标注的内容
    if os.path.exists(OUTPUT_FILE):
        print(f"发现已有输出文件 {OUTPUT_FILE}，正在合并已标注进度...")
        df_existing = pd.read_excel(OUTPUT_FILE)
        # 优先用 post_id 对齐，如果没有则用索引对齐（前提是原始数据行顺序未变）
        if "post_id" in df_raw.columns and "post_id" in df_existing.columns:
            cols_to_merge = ["post_id", "loss_type", "severity", "time_span", "llm_reason"]
            df_merge = df_existing[cols_to_merge]
            df_raw = df_raw.merge(df_merge, on="post_id", how="left", suffixes=("", "_old"))
            for col in ["loss_type", "severity", "time_span", "llm_reason"]:
                old_col = col + "_old"
                if old_col in df_raw.columns:
                    df_raw[col] = df_raw[old_col].combine_first(df_raw[col])
                    df_raw.drop(columns=[old_col], inplace=True)
            already_labeled = df_existing["loss_type"].notna().sum()
            print(f"已合并 {already_labeled} 条已标注数据。")
        else:
            # 无 post_id，简单按索引覆盖（确保行顺序没变过）
            for col in ["loss_type", "severity", "time_span", "llm_reason"]:
                if col in df_existing.columns:
                    df_raw[col] = df_existing[col]
            print("已按索引合并标注列（请确保原始数据行顺序未变）。")
    else:
        print("未找到已有输出文件，将从头开始标注。")
    return df_raw

def main():
    df = load_or_create_df()

    # 处理测试模式
    if TEST_MODE:
        df = df.head(20)
        print("⚠️ 测试模式开启，仅处理前20条。")

    # 确定哪些行需要标注（loss_type 为空 或 标注失败）
    need_process = (
        (df["loss_type"].isna()) |
        (df["loss_type"] == "") |
        (df["loss_type"] == "标注失败")
    )
    # 内容为空的行直接标记，不调用 API
    content_empty = (
        df[TEXT_COLUMN].isna() |
        (df[TEXT_COLUMN].astype(str).str.strip() == "") |
        (df[TEXT_COLUMN].astype(str) == "nan")
    )
    # 真正需要调用 API 的行
    to_process_mask = need_process & ~content_empty
    indices_to_process = df.index[to_process_mask].tolist()
    total_to_process = len(indices_to_process)

    print(f"待标注数据：{total_to_process} 条（已跳过内容为空的行）。")
    if total_to_process == 0:
        print("所有数据均已标注完成！")
        df.to_excel(OUTPUT_FILE, index=False)
        return

    # 开始逐条标注
    for i, idx in enumerate(tqdm(indices_to_process, desc="标注进度")):
        row = df.loc[idx]
        content = str(row[TEXT_COLUMN]) if pd.notna(row[TEXT_COLUMN]) else ""
        title = str(row[TITLE_COLUMN]) if TITLE_COLUMN in df.columns and pd.notna(row[TITLE_COLUMN]) else ""

        prompt = build_prompt(content, title)
        result = call_llm(prompt)

        df.at[idx, "loss_type"] = result.get("loss_type", "解析失败")
        df.at[idx, "severity"] = result.get("severity", "解析失败")
        df.at[idx, "time_span"] = result.get("time_span", "解析失败")
        df.at[idx, "llm_reason"] = result.get("brief_reason", "")

        time.sleep(SLEEP_TIME)

        # 定期保存进度
        if (i + 1) % SAVE_EVERY == 0:
            df.to_excel(OUTPUT_FILE, index=False)
            tqdm.write(f"已保存进度 ({i+1}/{total_to_process})")

    # 标记内容为空的行
    empty_indices = df.index[content_empty]
    for idx in empty_indices:
        if df.at[idx, "loss_type"] == "":
            df.at[idx, "loss_type"] = "内容为空"
            df.at[idx, "severity"] = "内容为空"
            df.at[idx, "time_span"] = "内容为空"
            df.at[idx, "llm_reason"] = "帖子内容为空"

    # 最终保存
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"\n✅ 全部完成！结果已保存至 {OUTPUT_FILE}")

if __name__ == "__main__":
    main()